# 04 · Collective consequences, forecasting, final report
**Topic 6 × Topic 5.** *H3:* λ raises the intraclass correlation ρ of correctness across i.i.d. samples, so K_eff = N/(1+(N−1)ρ) shrinks — but efficient models give more votes per token.
Where do depth–breadth token-budget frontiers cross? *H4:* side effects are forecastable early (curve extrapolation; early geometry → final drift).

In [ ]:
import sys, json, os, glob, time; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lab import profiles, local, envs, collective, cost
P = profiles.get(); MAIN = P["main"]; HF = {t: v[0] for t, v in P["models"].items()}; FINAL = P["steps"]
SPEC = f"spec_{P['name']}.json"
print("PROFILE:", P["name"], "| GPU:", P["gpu"], "| cap $", P["cap"], "| main model:", HF[MAIN], "| steps:", FINAL)

In [ ]:
spec = json.load(open(SPEC))
runs = sorted(k for k in json.load(open(local.runs_file())) if k.startswith(MAIN + "_lam") and k.endswith("_s0"))
models = [dict(tag="base", name="base", ckpt=None)] + [dict(tag=r, name=r, ckpt=FINAL) for r in runs]
sm = P["samples"]
local.launch("sample_many", dict(model=HF[MAIN], spec=spec, models=models, n_problems=sm["n_problems"], N=sm["N"], tag="coll"), key="samples_coll")
print(local.wait("samples", every=90)); local.pull("/samples")

In [ ]:
T = json.load(open(local.find("_truth.json")[0])); truth = T["truth"]; NP, N = len(truth), sm["N"]
def load(tag):
    d = pd.read_json(local.find(f"{tag}.jsonl", must="samples")[0], lines=True)
    C = np.zeros((NP, N), int); A = [[None] * N for _ in range(NP)]; Ln = np.zeros((NP, N))
    for r in d.itertuples(): C[r.p, r.j] = r.ok; A[r.p][r.j] = r.ans; Ln[r.p, r.j] = r.n
    return C, A, Ln
rows, data = [], {}
for m in models:
    C, A, Ln = load(m["tag"]); data[m["tag"]] = (C, A, Ln); rho = collective.icc_binary(C)
    rows.append(dict(model=m["tag"], acc=C.mean(), mean_len=Ln.mean(), ICC=rho, K_eff=collective.k_eff(N, rho),
                     wrong_mode_share=collective.wrong_mode_concentration(A, truth), pass_at_4=collective.pass_at_k(C, 4), maj=collective.maj_curve(A, truth, [N - 1], R=30)[0]))
pd.DataFrame(rows).round(3)

In [ ]:
ns = [n for n in (1, 2, 3, 5, 9, 15, 23, 31, 47, 63) if n < N]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for tag, (C, A, Ln) in data.items():
    y = collective.maj_curve(A, truth, ns, R=40); ax[0].plot(ns, y, "o-", label=tag); ax[1].plot(np.array(ns) * Ln.mean(), y, "o-")
ax[0].set_xlabel("n votes"); ax[0].set_ylabel("majority-vote accuracy"); ax[1].set_xscale("log"); ax[1].set_xlabel("total tokens per problem (n × mean length)")
ax[1].set_title("depth–breadth frontier: look for crossings"); ax[0].legend(fontsize=6); plt.tight_layout(); plt.show()

### Forecasting (H4a): extrapolate logged metrics from the first 40% of training vs a persistence baseline

In [ ]:
local.pull("/logs"); out = []; bin_ = max(1, FINAL // 20)
for r in runs:
    d = pd.read_json(local.find(f"{r}.jsonl", must="logs")[0], lines=True); dd = d.groupby(d.step // bin_).mean(numeric_only=True)
    for k in ("len", "acc", "trunc"):
        out.append(dict(run=r, metric=k, **collective.forecast_final(dd.index.values * bin_, dd[k].values, frac=0.4)))
F = pd.DataFrame(out); print(F.groupby("metric")[["err_fit", "err_persist"]].mean().round(4))

### Forecasting (H4b): early geometry → final drift (small n — report honestly)

In [ ]:
geo = json.load(open(local.find(f"geo_{MAIN}.json")[0])); mid = geo["layers"] // 2; E = pd.read_csv(f"evals_{P['name']}.csv"); base = E[E.run == "base"].iloc[0]
pts = []
for r in runs:
    e = E[(E.run == r) & (E.step == FINAL)]
    if len(e): pts.append(dict(run=r, early_proj_hum=geo["runs"][f"{r}_{P['early']}"]["proj"]["humility"][mid], early_proj_eff=geo["runs"][f"{r}_{P['early']}"]["proj"]["effort"][mid],
                               d_abstain=float(e.abstain_recall.iloc[0] - base.abstain_recall), d_flip_user=float(e.flip_user.iloc[0] - base.flip_user)))
Q = pd.DataFrame(pts); print(Q.round(3)); print(Q.drop(columns="run").corr(method="spearman").round(2))

### Report template
1. **Pareto:** length–accuracy frontier per λ; training-free `effort` steering overlaid.
2. **H1:** Δ disposition vs λ with bootstrap CIs, λ=0 RL control vs λ>0. Report nulls as nulls.
3. **H2:** projection table + steering×RL 2×2; state which outcome (mediates / reproduces / neither).
4. **H3:** ICC, K_eff, wrong-mode share vs λ; token-budget crossing point if any.
5. **H4:** forecast error vs persistence; geometry→drift Spearman (small n).
6. **Budget:** ledger below. Limitations: Qwen2.5 family, simplified persona vectors, programmatic probes.

In [ ]:
df, spent = local.ledger(); print(df.groupby("fn").usd.sum().round(2)); print(f"TOTAL SPENT ${spent:.2f} of ${P['cap']:.0f}")